# RouterR1 - Inference

This notebook demonstrates how to use **RouterR1** for agentic routing and reasoning.

## Overview

RouterR1 is an agentic router that performs R1-like reasoning with iterative search and routing.
It uses vLLM for local inference and an external routing API for model selection.

**Key Features**:
- Agentic reasoning with iterative search
- Pre-trained model (no training required)
- Uses vLLM for efficient GPU inference
- Integrates with external routing pool API

**Requirements**:
- CUDA-enabled GPU
- vLLM library
- OpenAI-compatible API endpoint

## 1. Environment Setup

In [1]:
# For Google Colab: Ensure GPU runtime is enabled
# Runtime -> Change runtime type -> Hardware accelerator -> GPU

import os

if 'COLAB_GPU' in os.environ:
    !git clone https://github.com/ulab-uiuc/LLMRouter.git
    %cd LLMRouter
    !pip install -e .
    !pip install vllm transformers pyyaml openai torch

In [2]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

In [3]:
import torch

# Check CUDA availability (required for RouterR1)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
else:
    print("WARNING: RouterR1 requires CUDA. Please enable GPU runtime.")

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU count: 1


In [4]:
from llmrouter.utils import setup_environment
import yaml

setup_environment()
print("Environment setup complete!")

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment setup complete!


## 2. Configuration

RouterR1 requires the following configuration parameters:

| Parameter | Description | Required |
|-----------|-------------|----------|
| `model_id` | HuggingFace model ID for reasoning | Yes |
| `api_base` | Routing API endpoint URL | Yes |
| `api_key` | API key for routing service | Yes |

**Note**: Create a config file or set these parameters programmatically.

In [5]:
# Create configuration for RouterR1
# Replace with your actual API credentials

router_r1_config = {
    "data_path": {
        "query_data_train": "data/example_data/query_data/default_query_train.jsonl",
        "query_data_test": "data/example_data/query_data/default_query_test.jsonl",
        "routing_data_train": "data/example_data/routing_data/default_routing_train_data.jsonl",
        "routing_data_test": "data/example_data/routing_data/default_routing_test_data.jsonl",
        "llm_data": "data/example_data/llm_candidates/default_llm.json"
    },
    "hparam": {
        "model_id": "Qwen/Qwen2.5-3B-Instruct",  # Pre-trained model for reasoning
        "api_base": "https://api.openai.com/v1",  # Replace with your API endpoint
        "api_key": os.environ.get("OPENAI_API_KEY", "your-api-key-here")  # Replace with your API key
    }
}

# Save config to temporary file
CONFIG_PATH = "configs/model_config_train/router_r1_temp.yaml"
os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(router_r1_config, f, default_flow_style=False)

print("Configuration:")
print("=" * 50)
print(yaml.dump(router_r1_config, default_flow_style=False))

Configuration:
data_path:
  llm_data: data/example_data/llm_candidates/default_llm.json
  query_data_test: data/example_data/query_data/default_query_test.jsonl
  query_data_train: data/example_data/query_data/default_query_train.jsonl
  routing_data_test: data/example_data/routing_data/default_routing_test_data.jsonl
  routing_data_train: data/example_data/routing_data/default_routing_train_data.jsonl
hparam:
  api_base: https://api.openai.com/v1
  api_key: your-key
  model_id: Qwen/Qwen2.5-3B-Instruct



## 3. Initialize Router

In [6]:
from llmrouter.models.router_r1 import RouterR1

# Initialize RouterR1 (requires valid API credentials)

try:
    router = RouterR1(yaml_path=CONFIG_PATH)
    print("RouterR1 initialized successfully!")
    print(f"Model ID: {router.model_id}")
except Exception as e:
    print(f"Error initializing RouterR1: {e}")
    print("Please ensure you have valid API credentials configured.")

✅ MetaRouter initialized successfully (YAML + data loaded).
RouterR1 initialized successfully!
Model ID: Qwen/Qwen2.5-3B-Instruct


## 4. Single Query Routing

RouterR1 performs agentic reasoning with iterative search and routing.
The process includes:
1. Initial prompt generation
2. Iterative reasoning with search queries
3. External routing API calls
4. Final answer generation

In [7]:
# Example query for RouterR1
test_query = {"query": "What is the capital of France and what is its population?"}

print(f"Query: {test_query['query']}")
print("=" * 60)

# Route with details (includes token counts)
try:
    result = router.route_single(test_query, return_details=True)
    
    print("\nResponse:")
    print(result["response"])
    print("\nToken Usage:")
    print(f"  Prompt tokens: {result['prompt_tokens']}")
    print(f"  Completion tokens: {result['completion_tokens']}")
    print(f"  Route tokens: {result['route_tokens']}")
    print(f"  Total tokens: {result['total_tokens']}")
except Exception as e:
    print(f"Error during routing: {e}")

Query: What is the capital of France and what is its population?
INFO 01-11 02:44:56 [utils.py:253] non-default args: {'dtype': 'float16', 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct'}
INFO 01-11 02:44:56 [model.py:514] Resolved architecture: Qwen2ForCausalLM
WARNING 01-11 02:44:56 [model.py:2005] Casting torch.bfloat16 to torch.float16.
INFO 01-11 02:44:56 [model.py:1661] Using max model len 32768


2026-01-11 02:45:00,944	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-11 02:45:00 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-11 02:45:01 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=1301) INFO 01-11 02:45:08 [core.py:93] Initializing a V1 LLM engine (v0.13.0) with config: model='Qwen/Qwen2.5-3B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=Str

(EngineCore_DP0 pid=1301) /opt/conda/lib/python3.13/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=1301) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=1301)   warnings.warn(


(EngineCore_DP0 pid=1301) INFO 01-11 02:45:11 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')


...

[Generation 0] Output:
<think>
To answer this question, we need to ascertain both the capital of France and its population. The capital of France is Paris. For the population, we need to find that specific information, as the population numbers can fluctuate over time. While specific population numbers can be found readily, we might not directly access them without an external search. To narrow down which LLM to consult, we should consider the most appropriate for handling this factual query: definitively knowing the capital and the population from a reliable source.
</think>

<search>LLaMA-3.1-8B-Instruct: The capital of France and its current population. </search>
meta/llama-3.1-8b-instruct


Processing: 100%|█████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.16s/it]


I can assist with the sub-question about the capital of France and its current population.

The capital of France is Paris. 

As for the current population of Paris, according to the latest available data (2020 estimates), the population of the city of Paris is approximately 2.1 million people. However, the population of the larger metropolitan area of Paris, which includes the surrounding suburbs and cities, is around 12.2 million people.

Please note that population figures may have changed slightly since the last available estimates. For the most up-to-date information, I recommend consulting a reliable source such as the French National Institute for Statistics and Economic Studies (INSEE) or the World Bank. 139


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.61it/s, est. speed input: 3511.09 toks/s, output: 136.24 toks/s]
[rank0]:[W111 02:46:11.281526755 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


[Generation 1] Output:
<answer>Paris is the capital of France. As of the latest available estimates, the current population of Paris is approximately 2.1 million people. The total population of the metropolitan area of Paris is around 12.2 million people.</answer>


################# [Output] ##################



<think>
To answer this question, we need to ascertain both the capital of France and its population. The capital of France is Paris. For the population, we need to find that specific information, as the population numbers can fluctuate over time. While specific population numbers can be found readily, we might not directly access them without an external search. To narrow down which LLM to consult, we should consider the most appropriate for handling this factual query: definitively knowing the capital and the population from a reliable source.
</think>

<search>LLaMA-3.1-8B-Instruct: The capital of France and its current population. </search>
<information>I can assist with t

## 5. Batch Routing

In [8]:
# Batch routing with multiple queries
batch_queries = [
    {"query": "What is machine learning?"},
    {"query": "Explain quantum computing in simple terms."},
    {"query": "How does photosynthesis work?"},
]

print(f"Processing {len(batch_queries)} queries...")
print("=" * 60)

try:
    results = router.route_batch(batch_queries)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. Query: {result.get('query', 'N/A')[:50]}...")
        print(f"   Success: {result.get('success', False)}")
        print(f"   Response length: {len(result.get('response', ''))} chars")
        print(f"   Tokens: {result.get('prompt_tokens', 0) + result.get('completion_tokens', 0)}")
except Exception as e:
    print(f"Error during batch routing: {e}")

Processing 3 queries...
INFO 01-11 02:46:21 [utils.py:253] non-default args: {'dtype': 'float16', 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct'}
INFO 01-11 02:46:22 [model.py:514] Resolved architecture: Qwen2ForCausalLM
WARNING 01-11 02:46:22 [model.py:2005] Casting torch.bfloat16 to torch.float16.
INFO 01-11 02:46:22 [model.py:1661] Using max model len 32768
INFO 01-11 02:46:22 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=1509) INFO 01-11 02:46:31 [core.py:93] Initializing a V1 LLM engine (v0.13.0) with config: model='Qwen/Qwen2.5-3B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None,

(EngineCore_DP0 pid=1509) /opt/conda/lib/python3.13/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=1509) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=1509)   warnings.warn(


(EngineCore_DP0 pid=1509) INFO 01-11 02:46:35 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:21<00:21, 21.58s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:33<00:00, 15.89s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:33<00:00, 16.74s/it]
(EngineCore_DP0 pid=1509) 


(EngineCore_DP0 pid=1509) INFO 01-11 02:47:09 [default_loader.py:308] Loading weights took 33.66 seconds
(EngineCore_DP0 pid=1509) INFO 01-11 02:47:10 [gpu_model_runner.py:3659] Model loading took 5.7916 GiB memory and 35.933024 seconds
(EngineCore_DP0 pid=1509) INFO 01-11 02:47:17 [backends.py:643] Using cache directory: /home/zhongjie/.cache/vllm/torch_compile_cache/01e8e4529a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1509) INFO 01-11 02:47:17 [backends.py:703] Dynamo bytecode transform time: 7.32 s
(EngineCore_DP0 pid=1509) INFO 01-11 02:47:26 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.490 s
(EngineCore_DP0 pid=1509) INFO 01-11 02:47:26 [monitor.py:34] torch.compile takes 8.81 s in total
(EngineCore_DP0 pid=1509) INFO 01-11 02:47:27 [gpu_worker.py:375] Available KV cache memory: 64.08 GiB
(EngineCore_DP0 pid=1509) INFO 01-11 02:47:27 [kv_cache_utils.py:1291] GPU KV cache size: 1,866,416 tokens
(Engine

...

[Generation 0] Output:
<think>
To answer the question about how photosynthesis works, I need to gather information about the process and its components. Photosynthesis is a multi-step process that plants and other organisms use to convert light energy into chemical energy, which are then stored in the form of organic compounds. Each of the selected large language models could potentially provide accurate information about this process, but let's consider the capabilities of the models mentioned. Given the need for accuracy and detail, LLaMA-3.1-70B-Instruct or Mistral-7B-Instruct might be particularly fitting due to their strong performance on standard industry benchmarks and tasks in understanding and generating complex texts.

I will now search for reliable information about photosynthesis using LLaMA-3.1-70B-Instruct.
</think>
<search>LLaMA-3.1-70B-Instruct: How does photosynthesis work? </search>
meta/llama-3.1-70b-instruct


Processing: 100%|█████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.06s/it]


I'd be happy to help explain how photosynthesis works.

**Direct Answer:** Photosynthesis is the process by which plants, algae, and some bacteria convert light energy from the sun into chemical energy in the form of organic compounds, such as glucose. This process occurs in specialized organelles called chloroplasts and involves the conversion of carbon dioxide and water into glucose and oxygen.

**Brief Explanation:** Photosynthesis involves two stages: the light-dependent reactions and the light-independent reactions (Calvin cycle). In the light-dependent reactions, light energy is absorbed by pigments such as chlorophyll and converted into ATP and NADPH. In the Calvin cycle, CO2 is fixed into glucose using the ATP and NADPH produced in the light-dependent reactions.

**Helpful Context:** Photosynthesis is essential for life on Earth, as it provides the energy and organic compounds needed to support the food chain. It's estimated that photosynthesis produces between 100 and 150 bill

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.72s/it, est. speed input: 892.88 toks/s, output: 136.11 toks/s]
[rank0]:[W111 02:50:07.049585785 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


[Generation 1] Output:
<answer>
Photosynthesis is the process by which plants, algae, and some bacteria convert light energy from the sun into chemical energy in the form of organic compounds, such as glucose. This process occurs in specialized organelles called chloroplasts and involves the conversion of carbon dioxide and water into glucose and oxygen. 

The process breaks down into two stages: the light-dependent reactions and the Calvin cycle. In the light-dependent reactions, light energy is absorbed by pigments such as chlorophyll and converted into ATP (adenosine triphosphate) and NADPH (nicotinamide adenine dinucleotide phosphate), energy-carrying molecules. In the Calvin cycle, CO2 is fixed into glucose using the ATP and NADPH produced in the light-dependent reactions. This cycle also generates oxygen as a byproduct.

Photosynthesis is essential for life on Earth, as it provides the energy and organic compounds needed to support the food chain. It's estimated that photosynthes

## 6. Task-Specific Routing

RouterR1 can format queries for specific tasks (MMLU, GSM8K, etc.).

In [9]:
# Task-specific routing with ground truth for evaluation
task_query = {
    "query": "What is the derivative of x^2?",
    "task_name": "math",
    "ground_truth": "2x"
}

print(f"Task Query: {task_query['query']}")
print(f"Task: {task_query['task_name']}")
print(f"Ground Truth: {task_query['ground_truth']}")
print("=" * 60)

try:
    results = router.route_batch([task_query], task_name="math")
    result = results[0]
    
    print(f"\nResponse: {result.get('response', 'N/A')[:200]}...")
    if 'task_performance' in result:
        print(f"Task Performance: {result['task_performance']:.2f}")
except Exception as e:
    print(f"Error: {e}")

Task Query: What is the derivative of x^2?
Task: math
Ground Truth: 2x
INFO 01-11 02:51:56 [utils.py:253] non-default args: {'dtype': 'float16', 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct'}
INFO 01-11 02:51:56 [model.py:514] Resolved architecture: Qwen2ForCausalLM
WARNING 01-11 02:51:56 [model.py:2005] Casting torch.bfloat16 to torch.float16.
INFO 01-11 02:51:56 [model.py:1661] Using max model len 32768
INFO 01-11 02:51:56 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=2110) INFO 01-11 02:52:06 [core.py:93] Initializing a V1 LLM engine (v0.13.0) with config: model='Qwen/Qwen2.5-3B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disa

(EngineCore_DP0 pid=2110) /opt/conda/lib/python3.13/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=2110) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=2110)   warnings.warn(


(EngineCore_DP0 pid=2110) INFO 01-11 02:52:10 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:20<00:20, 20.70s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:33<00:00, 15.99s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:33<00:00, 16.70s/it]
(EngineCore_DP0 pid=2110) 


(EngineCore_DP0 pid=2110) INFO 01-11 02:52:44 [default_loader.py:308] Loading weights took 33.61 seconds
(EngineCore_DP0 pid=2110) INFO 01-11 02:52:44 [gpu_model_runner.py:3659] Model loading took 5.7916 GiB memory and 35.955312 seconds
(EngineCore_DP0 pid=2110) INFO 01-11 02:52:52 [backends.py:643] Using cache directory: /home/zhongjie/.cache/vllm/torch_compile_cache/01e8e4529a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2110) INFO 01-11 02:52:52 [backends.py:703] Dynamo bytecode transform time: 7.16 s
(EngineCore_DP0 pid=2110) INFO 01-11 02:53:00 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.466 s
(EngineCore_DP0 pid=2110) INFO 01-11 02:53:00 [monitor.py:34] torch.compile takes 8.63 s in total
(EngineCore_DP0 pid=2110) INFO 01-11 02:53:01 [gpu_worker.py:375] Available KV cache memory: 64.08 GiB
(EngineCore_DP0 pid=2110) INFO 01-11 02:53:02 [kv_cache_utils.py:1291] GPU KV cache size: 1,866,416 tokens
(Engine

...

To find the derivative of \(x^2\), we can use the power rule of differentiation. The power rule states that if we have a function of the form \(x^n\), its derivative is \(nx^{n-1}\).

Applying this rule to \(x^2\), we get:

- The exponent \(n\) is 2.
- The derivative of \(x^2\) is therefore \(2x^{2-1}\).
- Simplifying the exponent, we get \(2x^1\).
- Since \(x^1\) is equivalent to \(x\), the derivative of \(x^2\) is \(2x\).

Therefore, the derivative of \(x^2\) is \(2x\). 153


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 10.84it/s, est. speed input: 14294.29 toks/s, output: 130.92 toks/s]

[Generation 1] Output:
</think>

<answer> 2x </answer>


################# [Output] ##################



<think>
The question asks for the derivative of \(x^2\). To provide a precise and educational response on the topic, using a specialized model like LLaMA-3.1-8B-Instruct, would be the most appropriate choice due to its strong reasoning capabilities and performance in various tasks including handling mathematical derivations.

<search>LLaMA-3.1-8B-Instruct: What is the derivative of \(x^2\) </search>
<information>To find the derivative of \(x^2\), we can use the power rule of differentiation. The power rule states that if we have a function of the form \(x^n\), its derivative is \(nx^{n-1}\).

Applying this rule to \(x^2\), we get:

- The exponent \(n\) is 2.
- The derivative of \(x^2\) is therefore \(2x^{2-1}\).
- Simplifying the exponent, we get \(2x^1\).
- Since \(x^1\) is equivalent to \(x\), the derivative of \(x^2\) is \(2x\).

Therefore, the derivative of \(x^2\) is \(2x\).</


[rank0]:[W111 02:53:09.710784467 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



Response: <think>
The question asks for the derivative of \(x^2\). To provide a precise and educational response on the topic, using a specialized model like LLaMA-3.1-8B-Instruct, would be the most appropriate...
Task Performance: 0.00


## 7. File-Based Inference

Load queries from a file and save results.

In [11]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "LLMRouter/data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/router_r1_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries (limit to 5 for demo due to API costs)
    try:
        file_results = router.route_batch(file_queries[:5])
        print(f"Routed {len(file_results)} queries")
        
        # Save results
        save_results_to_file(file_results, OUTPUT_FILE)
        
        # Show sample results
        print(f"\nSample results:")
        for i, result in enumerate(file_results[:3], 1):
            print(f"  {i}. {result.get('query', '')[:40]}...")
            print(f"     Success: {result.get('success', False)}")
    except Exception as e:
        print(f"Error during batch routing: {e}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

Loaded 706 queries from: LLMRouter/data/example_data/query_data/default_query_test.jsonl
INFO 01-11 02:53:42 [utils.py:253] non-default args: {'dtype': 'float16', 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct'}
INFO 01-11 02:53:42 [model.py:514] Resolved architecture: Qwen2ForCausalLM
WARNING 01-11 02:53:42 [model.py:2005] Casting torch.bfloat16 to torch.float16.
INFO 01-11 02:53:42 [model.py:1661] Using max model len 32768
INFO 01-11 02:53:42 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=2319) INFO 01-11 02:53:51 [core.py:93] Initializing a V1 LLM engine (v0.13.0) with config: model='Qwen/Qwen2.5-3B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_par

(EngineCore_DP0 pid=2319) /opt/conda/lib/python3.13/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=2319) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=2319)   warnings.warn(


(EngineCore_DP0 pid=2319) INFO 01-11 02:53:55 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:19<00:19, 19.71s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:34<00:00, 16.65s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:34<00:00, 17.11s/it]
(EngineCore_DP0 pid=2319) 


(EngineCore_DP0 pid=2319) INFO 01-11 02:54:30 [default_loader.py:308] Loading weights took 34.46 seconds
(EngineCore_DP0 pid=2319) INFO 01-11 02:54:30 [gpu_model_runner.py:3659] Model loading took 5.7916 GiB memory and 36.793859 seconds
(EngineCore_DP0 pid=2319) INFO 01-11 02:54:38 [backends.py:643] Using cache directory: /home/zhongjie/.cache/vllm/torch_compile_cache/01e8e4529a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=2319) INFO 01-11 02:54:38 [backends.py:703] Dynamo bytecode transform time: 7.13 s
(EngineCore_DP0 pid=2319) INFO 01-11 02:54:46 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.642 s
(EngineCore_DP0 pid=2319) INFO 01-11 02:54:46 [monitor.py:34] torch.compile takes 8.77 s in total
(EngineCore_DP0 pid=2319) INFO 01-11 02:54:47 [gpu_worker.py:375] Available KV cache memory: 64.08 GiB
(EngineCore_DP0 pid=2319) INFO 01-11 02:54:48 [kv_cache_utils.py:1291] GPU KV cache size: 1,866,416 tokens
(Engine

...

(EngineCore_DP0 pid=3093) INFO 01-11 03:00:16 [default_loader.py:308] Loading weights took 38.34 seconds
(EngineCore_DP0 pid=3093) INFO 01-11 03:00:16 [gpu_model_runner.py:3659] Model loading took 5.7916 GiB memory and 40.728790 seconds
(EngineCore_DP0 pid=3093) INFO 01-11 03:00:24 [backends.py:643] Using cache directory: /home/zhongjie/.cache/vllm/torch_compile_cache/01e8e4529a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3093) INFO 01-11 03:00:24 [backends.py:703] Dynamo bytecode transform time: 7.47 s
(EngineCore_DP0 pid=3093) INFO 01-11 03:00:33 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.738 s
(EngineCore_DP0 pid=3093) INFO 01-11 03:00:33 [monitor.py:34] torch.compile takes 9.21 s in total
(EngineCore_DP0 pid=3093) INFO 01-11 03:00:36 [gpu_worker.py:375] Available KV cache memory: 64.08 GiB
(EngineCore_DP0 pid=3093) INFO 01-11 03:00:37 [kv_cache_utils.py:1291] GPU KV cache size: 1,866,416 tokens
(Engine

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 21.28it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 22.81it/s]


(EngineCore_DP0 pid=3093) INFO 01-11 03:00:42 [gpu_model_runner.py:4587] Graph capturing finished in 5 secs, took 0.58 GiB
(EngineCore_DP0 pid=3093) INFO 01-11 03:00:42 [core.py:259] init engine (profile, create kv cache, warmup model) took 25.62 seconds
INFO 01-11 03:00:43 [llm.py:360] Supported tasks: ['generate']


################# [Start Reasoning + Routing] ##################




Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.14s/it, est. speed input: 217.50 toks/s, output: 136.69 toks/s]
[rank0]:[W111 03:00:50.412016375 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


[Generation 0] Output:
To determine the house where the person who drives a truck lives, I need to gather information about the distribution of the cars and their corresponding locations based on the given clues.

### Step-by-step reasoning:

First, let's denote people with the different characteristics:
1. Person A: drives a convertible
2. Person B: drives a minivan
3. Person C: drives a truck
4. Person D: ate grilled cheese
5. Person E: ate spaghetti
6. Person F: had stew
7. Person G: likes blue
8. Person H: likes red
9. Person I: has blue
10. Person J: has a rose bouquet
11. Person K: has a bouquet of daffodils
12. Person L: has a bouquet of lilies

### Clue Analysis:

#### Clue 1: The person who likes blue lives directly left of the person who ate spaghetti.
- Someone (I) likes blue (blue).
- Someone (J) ate spaghetti (spaghetti).

Possible positions for I and J:
- (I, J): blue then spaghetti
- (J, I): spaghetti then blue

#### Clue 2: The person who had stew lives somewhere to the

## Summary

**RouterR1 Key Points**:
- Pre-trained agentic router (no training required)
- Uses vLLM for efficient GPU inference
- Iterative reasoning with external routing API
- Supports task-specific query formatting
- Token tracking for cost analysis

**Requirements**:
- CUDA-enabled GPU (vLLM requirement)
- Valid API credentials for routing service
- Sufficient GPU memory for the base model

**Use Cases**:
- Complex queries requiring reasoning
- Multi-step question answering
- Research and evaluation on routing strategies